In [ ]:
# ==== CONFIG ====
TICKERS   = ["AAPL","MSFT","AMZN","GOOGL","NVDA"]  # same universe as your price-only model
START     = "2012-01-01"
END       = "2022-12-31"

SEQ_LEN   = 20           # shorter window for classification; tune later (10/20/60)
THRESH    = 0.005        # 0.5% up/down threshold → 3-class
BATCH     = 64
EPOCHS    = 30
PATIENCE  = 6
LR        = 1e-3

# If you already computed daily sentiment per (date, stock), point here:
NEWS_DAILY_PATH = "../data/Daily-Financial-News/news_sentiment_daily.csv"
# Expect columns (at least): date, stock, sent_mean, sent_std, prob_pos, prob_neg, news_count


In [ ]:
import os, math, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


In [ ]:
def SMA(s, n): return s.rolling(n).mean()
def EMA(s, n): return s.ewm(span=n, adjust=False).mean()

def build_features(df_px):
    df = df_px.copy()
    # Handle MultiIndex (yfinance sometimes returns ('close','aapl') style)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[-1].capitalize() for c in df.columns]

    # Normalize common names
    rename_map = {c: c.capitalize() for c in df.columns}
    df = df.rename(columns=rename_map)

    # Require these columns
    need = {"Open","High","Low","Close","Volume"}
    assert need.issubset(df.columns), f"Missing OHLCV columns: have {df.columns}"

    # Base returns + technicals
    df["Return"] = df["Close"].pct_change()
    df["LogRet"] = np.log1p(df["Return"])

    df["SMA_10"]  = SMA(df["Close"], 10)
    df["SMA_20"]  = SMA(df["Close"], 20)
    df["EMA_10"]  = EMA(df["Close"], 10)
    df["EMA_20"]  = EMA(df["Close"], 20)

    delta = df["Close"].diff()
    up, down = delta.clip(lower=0), -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1/14, adjust=False).mean()
    roll_down = down.ewm(alpha=1/14, adjust=False).mean()
    rs = roll_up / (roll_down + 1e-9)
    df["RSI_14"] = 100 - (100 / (1 + rs))

    ema12 = EMA(df["Close"], 12)
    ema26 = EMA(df["Close"], 26)
    df["MACD"] = ema12 - ema26
    df["MACDsig"] = EMA(df["MACD"], 9)

    bb_mid = SMA(df["Close"], 20)
    bb_std = df["Close"].rolling(20).std()
    df["BB_up"] = bb_mid + 2*bb_std
    df["BB_dn"] = bb_mid - 2*bb_std

    return df

def label_direction(df, thr=0.005):
    # next-day return
    r = df["Close"].pct_change().shift(-1).to_numpy().reshape(-1)  # 1D
    lab = np.full(len(df), 1, dtype=np.int64)
    lab[r >  thr] = 2
    lab[r < -thr] = 0
    df["Label"] = lab
    return df

PRICE_FEATURES = [
    "Open","High","Low","Close","Volume",
    "Return","LogRet","SMA_10","SMA_20","EMA_10","EMA_20",
    "RSI_14","MACD","MACDsig","BB_up","BB_dn"
]


In [ ]:
def load_news_daily(path):
    if not os.path.exists(path):
        print(f"[WARN] {path} not found. Using zero news features (pipeline sanity check).")
        return None

    news = pd.read_csv(path)
    # normalize col names
    news.columns = [c.strip().lower() for c in news.columns]
    # required columns
    required = {"date","stock"}
    if not required.issubset(set(news.columns)):
        raise ValueError(f"news file must have columns at least {required}")

    # suggested feature cols; we'll keep only what's available
    candidate_feats = ["sent_mean","sent_std","prob_pos","prob_neg","news_count"]
    avail = [c for c in candidate_feats if c in news.columns]
    if not avail:
        print("[WARN] no sentiment columns found — using zeros.")
        return None

    news = news[["date","stock"] + avail].copy()
    # types
    news["date"] = pd.to_datetime(news["date"], errors="coerce")
    news["stock"] = news["stock"].astype(str).str.upper().str.strip()
    # lag by 1 trading day per stock (so we don't leak same-day info)
    news = news.sort_values(["stock","date"])
    for c in avail:
        news[f"{c}_lag1"] = news.groupby("stock")[c].shift(1)
    keep = ["date","stock"] + [f"{c}_lag1" for c in avail]
    news = news[keep].dropna().copy()
    return news

news_daily = load_news_daily(NEWS_DAILY_PATH)

# Final news feature column names we’ll try to merge
NEWS_FEATS_LAG = []
if news_daily is not None:
    NEWS_FEATS_LAG = [c for c in news_daily.columns if c.endswith("_lag1") and c not in ("date","stock")]
    print("News features (lagged) found:", NEWS_FEATS_LAG)
else:
    print("Proceeding with zero news features.")


In [ ]:
frames = []
for t in TICKERS:
    px = yf.download(t, start=START, end=END, progress=False, group_by="ticker")
    if px is None or px.empty:
        print(f"[WARN] No data for {t}")
        continue

    # If MultiIndex style, pick this ticker's subframe
    if isinstance(px.columns, pd.MultiIndex):
        # yfinance makes columns like ('Close','AAPL'); convert to single-level
        px.columns = [c[0].capitalize() for c in px.columns]

    df = build_features(px)
    df["Date"] = df.index

    # label next-day movement
    df = label_direction(df, thr=THRESH)

    # Drop rows where price features are still NaN (from rolling windows)
    df = df.dropna(subset=PRICE_FEATURES + ["Label"]).copy()

    df["Ticker"] = t
    frames.append(df)

assert frames, "No price data gathered."
prices_all = pd.concat(frames, axis=0).reset_index(drop=True)

# Merge with lagged news on (date, stock==ticker)
if news_daily is not None:
    merged = prices_all.merge(
        news_daily.rename(columns={"stock":"Ticker","date":"Date"}),
        on=["Ticker","Date"],
        how="left"
    )
    # Fill missing news (days with no news) with zeros
    for c in NEWS_FEATS_LAG:
        merged[c] = merged[c].fillna(0.0)
else:
    merged = prices_all.copy()

merged = merged.sort_values(["Ticker","Date"]).reset_index(drop=True)
print("Merged shape:", merged.shape)
merged.head(3)[["Ticker","Date","Close","Label"] + (NEWS_FEATS_LAG[:3] if NEWS_FEATS_LAG else [])]


In [ ]:
# Chronological split by Date across all tickers (like your other models)
dates_sorted = np.sort(merged["Date"].unique())
d_train = dates_sorted[int(0.70*len(dates_sorted)) - 1]
d_val   = dates_sorted[int(0.85*len(dates_sorted)) - 1]

train_df = merged[merged["Date"] <= d_train].copy()
val_df   = merged[(merged["Date"] > d_train) & (merged["Date"] <= d_val)].copy()
test_df  = merged[merged["Date"] > d_val].copy()

print("Split dates:")
print("  Train  ≤", pd.to_datetime(d_train).date())
print("  Val    >", pd.to_datetime(d_train).date(), "and ≤", pd.to_datetime(d_val).date())
print("  Test   >", pd.to_datetime(d_val).date())
print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train label balance:", train_df["Label"].value_counts(normalize=True).sort_index())

# Build final feature list
ALL_FEATURES = PRICE_FEATURES + NEWS_FEATS_LAG
print(f"Using {len(ALL_FEATURES)} features:", ALL_FEATURES)

# Fit scaler on TRAIN ONLY
scaler = StandardScaler().fit(train_df[ALL_FEATURES].values)

def create_sequences_per_ticker(df_split, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for t in df_split["Ticker"].unique():
        g = df_split[df_split["Ticker"] == t].sort_values("Date")
        if len(g) < seq_len + 1: 
            continue
        X_scaled = scaler.transform(g[ALL_FEATURES].values)
        y = g["Label"].values.astype(np.int64)
        for i in range(seq_len, len(g)):
            Xs.append(X_scaled[i-seq_len:i])
            ys.append(y[i])
    if len(Xs) == 0:
        return np.empty((0, seq_len, len(ALL_FEATURES)), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int64)

X_tr, y_tr = create_sequences_per_ticker(train_df, SEQ_LEN)
X_va, y_va = create_sequences_per_ticker(val_df,   SEQ_LEN)
X_te, y_te = create_sequences_per_ticker(test_df,  SEQ_LEN)

print("Shapes:")
print("  Train:", X_tr.shape, y_tr.shape)
print("  Val:  ", X_va.shape, y_va.shape)
print("  Test: ", X_te.shape, y_te.shape)
print("Test label balance:", np.bincount(y_te)/len(y_te))


In [ ]:
class SeqDS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i])

train_loader = DataLoader(SeqDS(X_tr, y_tr), batch_size=BATCH, shuffle=True, drop_last=True)
val_loader   = DataLoader(SeqDS(X_va, y_va), batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(SeqDS(X_te, y_te), batch_size=BATCH, shuffle=False)

class PriceNewsLSTM(nn.Module):
    def __init__(self, n_features, hidden=128, layers=2, dropout=0.25, bidirectional=True, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout if layers>1 else 0.0,
            bidirectional=bidirectional
        )
        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Dropout(dropout),
            nn.Linear(out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last)

model = PriceNewsLSTM(n_features=len(ALL_FEATURES)).to(DEVICE)
model


In [ ]:
# Class weights to help with imbalance
class_counts = np.bincount(y_tr, minlength=3).astype(np.float32)
w = (class_counts.sum() / (class_counts + 1e-9))
w = w / w.mean()
print("Class weights:", w)

criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

def run_epoch(loader, train=True):
    model.train(train)
    total, preds_all, labels_all = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total += loss.item() * xb.size(0)
        preds_all.append(logits.detach().cpu().numpy())
        labels_all.append(yb.detach().cpu().numpy())
    preds = np.argmax(np.vstack(preds_all), axis=1)
    labels= np.hstack(labels_all)
    acc = (preds == labels).mean()
    f1m = f1_score(labels, preds, average="macro")
    return total/len(loader.dataset), acc, f1m

best_val, patience = 1e9, 0
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader, False)
    scheduler.step(va_loss)
    improved = va_loss < best_val - 1e-5
    if improved:
        best_val = va_loss; patience = 0
        torch.save(model.state_dict(), "best_price_news_lstm.pth")
    else:
        patience += 1
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.3f} f1 {tr_f1:.3f}  |  "
              f"val loss {va_loss:.4f} acc {va_acc:.3f} f1 {va_f1:.3f}")
    if patience >= PATIENCE:
        print("Early stopping."); break

model.load_state_dict(torch.load("best_price_news_lstm.pth", map_location=DEVICE))
model.eval()

def infer(loader):
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            pr = model(xb.to(DEVICE))
            preds = torch.argmax(pr, dim=1).cpu().numpy()
            preds_all.append(preds)
            labels_all.append(yb.numpy())
    return np.hstack(preds_all), np.hstack(labels_all)

y_pred, y_true = infer(test_loader)
acc  = accuracy_score(y_true, y_pred)
f1m  = f1_score(y_true, y_pred, average="macro")
print(f"TEST — acc {acc:.3f}  macroF1 {f1m:.3f}")
print(classification_report(y_true, y_pred, target_names=["Down","Neutral","Up"]))


In [ ]:
def per_ticker_counts(df_split):
    counts = {}
    for t in df_split["Ticker"].unique():
        g = df_split[df_split["Ticker"] == t].sort_values("Date")
        n = max(0, len(g) - SEQ_LEN)
        counts[t] = n
    return counts

# rebuild test sequences order to slice correctly
def seq_counts(df_split, seq_len=SEQ_LEN):
    counts = {}
    for t in df_split["Ticker"].unique():
        n = max(0, len(df_split[df_split["Ticker"] == t]) - seq_len)
        counts[t] = n
    return counts

counts = seq_counts(test_df, SEQ_LEN)
idx = 0
perf = {}
for t in test_df["Ticker"].unique():
    n = counts.get(t, 0)
    if n <= 0: 
        continue
    preds = y_pred[idx:idx+n]
    true  = y_true[idx:idx+n]
    perf[t] = 100.0 * (preds == true).mean()
    idx += n

print("Per-ticker test accuracy:")
for k, v in sorted(perf.items(), key=lambda x: x[1], reverse=True):
    print(f"  {k}: {v:.1f}%")
